# 🚕 NYC Taxi Fare - Full Year Model Training (2024)
This notebook scales up our pipeline to process **all 12 months** of 2024 TLC data (approx 40 million records). It iterates through the datasets, aggressively optimizes memory footprints using downcasting, performs `RandomizedSearchCV` for hyperparameter tuning, and exports the master XGBoost model.\n

In [10]:
# Run this cell if you hit ModuleNotFoundError
%pip install pandas numpy xgboost scikit-learn pyarrow requests

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
import urllib.request
import gc
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Dynamic Path Resolution
# ---------------------------------------------------------
# If running in notebooks/, data is in ../service/ml-inference/data
# If running in root, data is in service/ml-inference/data
if Path('service/ml-inference/data').exists():
    DATA_DIR = Path('service/ml-inference/data')
elif Path('../service/ml-inference/data').exists():
    DATA_DIR = Path('../service/ml-inference/data')
else:
    raise FileNotFoundError("Could not locate the data directory. Please ensure you are running from the project root or the notebooks folder.")



## 1. ⬇️ Download the 12 Months of Data

In [12]:
def download_month(month: int):
    filename = f"yellow_tripdata_2024-{month:02d}.parquet"
    filepath = DATA_DIR / filename
    
    if filepath.exists():
        return filepath
        
    print(f"Downloading {filename} (~50MB)...")
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
    
    try:
        urllib.request.urlretrieve(url, filepath)
    except Exception as e:
        print(f"⚠️ Month {month} not available yet or link broken.")
        return None
        f.write(chunk)
    return filepath

parquet_files = []
for m in range(1, 13):
    fp = download_month(m)
    if fp:
        parquet_files.append(fp)

print(f"✅ Ready to process {len(parquet_files)} months of data.")


✅ Ready to process 12 months of data.


## 2. 🧮 Iterative Preprocessing Engine
Because 40 million rows can crash memory, we read, process, and compress each month individually before concatenating them.\n

In [13]:
# Load dimension tables once
dim_location = pd.read_csv(DATA_DIR / "dim_localizacao.csv")
dim_time = pd.read_csv(DATA_DIR / "dim_data.csv")
dim_weather = pd.read_csv(DATA_DIR / "dim_clima.csv")

dim_location.rename(columns={'localizacao_id': 'LocationID', 'bairro': 'Borough'}, inplace=True)
dim_time.rename(columns={'data_id': 'date_id', 'data': 'date', 'dia_semana': 'day_of_week'}, inplace=True)
dim_weather.rename(columns={'clima_id': 'weather_id', 'data': 'date', 'temp_max': 'temp_max', 'temp_min': 'temp_min', 'precipitacao': 'precip', 'neve': 'snow'}, inplace=True)

# Precompute the weather temperature
dim_weather['temp'] = ((dim_weather['temp_max'] + dim_weather['temp_min']) / 2.0).astype('float32')

def process_chunk(filepath):
    df = pd.read_parquet(filepath)
    
    # 1. Datetimes & High-Value Targets
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_time'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()
    df['request_hour'] = df['tpep_pickup_datetime'].dt.hour
    df['date'] = df['tpep_pickup_datetime'].dt.strftime('%Y-%m-%d')
    df.rename(columns={'trip_distance': 'trip_miles'}, inplace=True)
    
    # 2. Strict Filtering
    df = df[(df['trip_miles'] > 0.1) & (df['trip_miles'] < 100)]
    df = df[(df['fare_amount'] > 2.50) & (df['fare_amount'] < 400)]
    df = df[(df['trip_time'] > 60)] # Drop ghost trips
    
    # 3. Merge Dimensions
    df = df.merge(dim_location[['LocationID', 'Borough']], left_on="PULocationID", right_on="LocationID", how="inner").rename(columns={'Borough': 'PU_Borough'}).drop(columns=['LocationID'])
    df = df.merge(dim_location[['LocationID', 'Borough']], left_on="DOLocationID", right_on="LocationID", how="inner").rename(columns={'Borough': 'DO_Borough'}).drop(columns=['LocationID'])
    df = df.merge(dim_time[['date', 'day_of_week']], on="date", how="left")
    df = df.merge(dim_weather[['date', 'temp', 'precip', 'snow']], on="date", how="left")
    
    # Drop missing weather/fares
    df.dropna(subset=['fare_amount', 'trip_miles', 'temp'], inplace=True)
    
    # 4. Feature Engineering
    df['route_id'] = (df['PULocationID'] * 1000) + df['DOLocationID']
    airport_zones = [132, 138, 1]
    df['is_airport_trip'] = (df['PULocationID'].isin(airport_zones) | df['DOLocationID'].isin(airport_zones)).astype('int8')
    df['is_weekend'] = df['day_of_week'].isin([1, 7]).astype('int8')
    
    # Speed bounds
    df['implied_speed_mph'] = ((df['trip_miles'] / df['trip_time']) * 3600).clip(lower=0, upper=75).fillna(0).astype('float32')
    
    # 5. One Hot Encode Boroughs EARLY so we can cast to uint8
    df = pd.get_dummies(df, columns=['PU_Borough', 'DO_Borough'], drop_first=False, dtype='uint8')
    
    # 6. Strict Downcasting for RAM preservation
    cols_to_keep = ['tpep_pickup_datetime', 'fare_amount', 'trip_miles', 'trip_time', 
                    'request_hour', 'is_weekend', 'is_airport_trip', 'route_id',
                    'temp', 'precip', 'snow', 'implied_speed_mph']
    
    # Add all dummy columns
    borough_cols = [c for c in df.columns if 'PU_Borough_' in c or 'DO_Borough_' in c]
    cols_to_keep.extend(borough_cols)
    
    df = df[cols_to_keep]
    
    df['fare_amount'] = df['fare_amount'].astype('float32')
    df['trip_miles'] = df['trip_miles'].astype('float32')
    df['trip_time'] = df['trip_time'].astype('int32')
    df['request_hour'] = df['request_hour'].astype('int8')
    df['route_id'] = df['route_id'].astype('int32')
    df['precip'] = df['precip'].astype('float32')
    df['snow'] = df['snow'].astype('float32')
    
    return df

In [14]:
print("⏳ Processing chunks...")
processed_chunks = []
for idx, filepath in enumerate(parquet_files):
    print(f"   Processing Month {idx+1}...")
    chunk = process_chunk(filepath)
    processed_chunks.append(chunk)

print("🚀 Concatenating all months into master dataframe...")
master_df = pd.concat(processed_chunks, ignore_index=True)

# Free up memory
del processed_chunks
gc.collect()

# Sort Chronologically
master_df.sort_values('tpep_pickup_datetime', inplace=True)
master_df.reset_index(drop=True, inplace=True)

print(f"✅ Full Year Dataset Ready. Total Rows: {len(master_df):,}")

⏳ Processing chunks...
   Processing Month 1...
   Processing Month 2...
   Processing Month 3...
   Processing Month 4...
   Processing Month 5...
   Processing Month 6...
   Processing Month 7...
   Processing Month 8...
   Processing Month 9...
   Processing Month 10...
   Processing Month 11...
   Processing Month 12...
🚀 Concatenating all months into master dataframe...
✅ Full Year Dataset Ready. Total Rows: 3,949,995


## 3. ✂️ Chronological Splitting & Leakage-Free Target Encoding

In [15]:
# API Pydantic Alignment defaults
master_df['shared_match_flag'] = np.int8(0)
master_df['wav_match_flag'] = np.int8(0)
master_df['access_a_ride_flag'] = np.int8(0)

features = [
    'trip_miles', 'trip_time',
    'shared_match_flag', 'wav_match_flag', 'access_a_ride_flag',
    'request_hour', 'is_weekend', 'is_airport_trip', 
    'temp', 'precip', 'snow', 'implied_speed_mph'
]
borough_cols = [c for c in master_df.columns if 'PU_Borough_' in c or 'DO_Borough_' in c]
features.extend(borough_cols)

# Chronological 80/20 Split
split_idx = int(len(master_df) * 0.8)
train_df = master_df.iloc[:split_idx].copy()
test_df = master_df.iloc[split_idx:].copy()

# ---------------------------------------------------------
# Bayesian Smoothed Target Encoding on TRAINING SET ONLY
# ---------------------------------------------------------
route_stats = train_df.groupby('route_id')['fare_amount'].agg(['mean', 'count'])
global_mean = train_df['fare_amount'].mean()

m = 50 
route_stats['smoothed_mean'] = (
    (route_stats['count'] * route_stats['mean']) + (m * global_mean)
) / (route_stats['count'] + m)

route_means = route_stats['smoothed_mean'].astype('float32').to_dict()

# Map the values back
train_df['historical_route_fare'] = train_df['route_id'].map(route_means).astype('float32')
test_df['historical_route_fare'] = test_df['route_id'].map(route_means).fillna(global_mean).astype('float32')

features.append('historical_route_fare')

X_train = train_df[features]
y_train = train_df['fare_amount']
X_test = test_df[features]
y_test = test_df['fare_amount']

print(f"✅ Split Complete! Training Rows: {len(X_train):,}, Testing Rows: {len(X_test):,}")

# Memory cleanup
del master_df
gc.collect()

✅ Split Complete! Training Rows: 3,159,996, Testing Rows: 789,999


0

## 4. 🎛️ Hyperparameter Tuning (RandomizedSearchCV)
Tuning on ~30 million rows is extremely slow. We sample 500,000 rows to quickly find the best parameters.

In [16]:
# Create a 500k sample
sample_indices = np.random.choice(len(X_train), size=min(500000, len(X_train)), replace=False)
X_train_sample = X_train.iloc[sample_indices]
y_train_sample = y_train.iloc[sample_indices]

# Define the parameter grid
param_grid = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [6, 8, 10],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Use TimeSeriesSplit because this is chronological data
tscv = TimeSeriesSplit(n_splits=3)

xgb_model = xgb.XGBRegressor(tree_method='hist', random_state=42, n_jobs=-1)

# RandomizedSearch to save time
print("🔍 Running RandomizedSearchCV on 500k sample... (This may take a few minutes)")
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=10,  # Try 10 random combinations
    scoring='neg_mean_absolute_error',
    cv=tscv,
    verbose=1,
    n_jobs=1,
    random_state=42
)

search.fit(X_train_sample, y_train_sample)
best_params = search.best_params_
print(f"🏆 Best Parameters Found: {best_params}")


🔍 Running RandomizedSearchCV on 500k sample... (This may take a few minutes)
Fitting 3 folds for each of 10 candidates, totalling 30 fits
🏆 Best Parameters Found: {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


## 5. 🧠 Final Master Training & Evaluation

In [17]:
print(f"🚀 Training Final Model on ALL {len(X_train):,} rows...")
final_model = xgb.XGBRegressor(
    **best_params,
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

final_model.fit(X_train, y_train)
print("✅ Training complete.")

print("📊 Evaluating Accuracy on Holdout Test Set...")
predictions = final_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mape = mean_absolute_percentage_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"🌟 Final Master Model MAE: ${mae:.2f}")
print(f"🌟 Final Master Model RMSE: ${rmse:.2f}")
print(f"🌟 Final Master Model MAPE: {mape:.2%}")
print(f"🌟 Final Master Model R²: {r2:.4f}")


🚀 Training Final Model on ALL 3,159,996 rows...
✅ Training complete.
📊 Evaluating Accuracy on Holdout Test Set...
🌟 Final Master Model MAE: $1.04
🌟 Final Master Model RMSE: $3.27
🌟 Final Master Model MAPE: 5.72%
🌟 Final Master Model R²: 0.9614


## 6. 💾 Exporting Production Artifacts

In [18]:
import json

# Export Model
model_path = DATA_DIR.parent / "production_taxi_master_model_2024.joblib"
joblib.dump(final_model, model_path)
print(f"✅ Model saved to {model_path}")

# Export Route Means for Inference API
route_means_path = DATA_DIR.parent / "route_means.json"
export_means = {str(k): float(v) for k, v in route_means.items()}
with open(route_means_path, 'w') as f:
    json.dump(export_means, f)
print(f"✅ Route means saved to {route_means_path}")


✅ Model saved to ../service/ml-inference/production_taxi_master_model_2024.joblib
✅ Route means saved to ../service/ml-inference/route_means.json
